<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/visualize_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!cd cursivetransformer && pip install -r requirements.txt
!wandb login

Cloning into 'cursivetransformer'...
remote: Enumerating objects: 3025, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 3025 (delta 487), reused 424 (delta 398), pack-reused 2479 (from 1)
Receiving objects: 100% (3025/3025), 64.43 MiB | 38.88 MiB/s, done.
Resolving deltas: 100% (1684/1684), done.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-56_xbu8t
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-56_xbu8t
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
wandb: Currently logged in as: zwimpee (llm-lab). Use `wandb login --relogin` to force relogin


In [5]:
import sys
sys.path.append('/content/cursivetransformer/')

import torch
import matplotlib.pyplot as plt
import seaborn as sns

from cursivetransformer.model import get_all_args, get_checkpoint
from cursivetransformer.data import create_datasets
from cursivetransformer.sample import generate_n_words, plot_strokes

In [ ]:
args = get_all_args(False)
args.wandb_project = 'bigbank_2k'
args.load_from_run_id = '7e9hz1og'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.vocab_size = train_dataset.get_vocab_size()
args.context_vocab_size = train_dataset.get_char_vocab_size()
print(f"Dataset determined that: {args.vocab_size=}, {args.block_size=}")
model, _, _, _, _ = get_checkpoint(args, sample_only=True)

Trying to load dataset file from /content/cursivetransformer/data/bigbank.json.zip
Succeeded in loading the bigbank dataset; contains 2000 items.
For a dataset of 1900 examples we can generate 205257574037880 combinations of 5 examples.
Generating 497000 random combinations.
For a dataset of 100 examples we can generate 75287520 combinations of 5 examples.
Generating 3000 random combinations.


In [ ]:
# Generate a sample
text = "Hello world"
n_words = 2
temperature = 1.0
top_k = None
do_sample = True
num_steps = 1250

offset_samp, point_samp, attention_patterns = generate_n_words(model, test_dataset, text, n_words, temperature, top_k, do_sample, num_steps, return_attention_patterns=True)

# Plot the generated strokes
plot_strokes(point_samp, title=text)

In [ ]:
def plot_attention_patterns(attention_patterns, layer, head):
    # Extract attention patterns for the specified layer and head
    layer_patterns = [step[f'transformer.h.{layer}.attn.hook_pattern'] for step in attention_patterns]
    head_patterns = [pattern[0, head] for pattern in layer_patterns]
    
    # Create a heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(head_patterns, cmap='viridis')
    plt.title(f'Attention Patterns for Layer {layer}, Head {head}')
    plt.xlabel('Token Position (Key)')
    plt.ylabel('Generation Step (Query)')
    plt.show()

# Plot attention patterns for a specific layer and head
plot_attention_patterns(attention_patterns, layer=0, head=0)

In [ ]:
num_layers = model.config.n_layer
num_heads = model.config.n_ctx_head

fig, axes = plt.subplots(num_layers, num_heads, figsize=(20, 5*num_layers))
fig.suptitle('Attention Patterns Across All Layers and Heads', fontsize=16)

for layer in range(num_layers):
    for head in range(num_heads):
        layer_patterns = [step[f'transformer.h.{layer}.attn.hook_pattern'] for step in attention_patterns]
        head_patterns = [pattern[0, head] for pattern in layer_patterns]
        
        ax = axes[layer, head]
        sns.heatmap(head_patterns, cmap='viridis', ax=ax, cbar=False)
        ax.set_title(f'L{layer}H{head}')
        ax.axis('off')

plt.tight_layout()
plt.show()